# Paso 3 - Deteccion de candidatos (100% desde cero)

**Regla de esta linea de trabajo (ver README.md):** sin apoyarse en
`shape_features_v2.py`, `comp_stats_v2.json`, anotaciones existentes, ni
ningun hallazgo de Ciclo Alpha v1/v2. Todo lo de abajo se construye solo con
lo que Investigacion_v2 ya calculo (Baseline v0/v1, `paso_estable`).

## Definicion: segmento activo = complemento de `paso_estable`

Paso 1 ya definio `paso_estable = (delta_peso == 0) & (~is_gap)`. Un
**segmento activo** es una corrida de lecturas CONSECUTIVAS donde hubo
movimiento real. Este notebook construye la definicion en dos pasadas: una
primera version ingenua, y una correccion encontrada al revisarla (ver
Hallazgo 6 mas abajo) -- se deja el proceso completo, no solo el resultado
final, porque el error de la v1 es sutil y vale la pena que quede visible.

Carga desde `data/lecturas_limpias.csv` (cache de Paso 1) -- **correr
`01_caracterizacion_fondo.ipynb` primero** si el cache no existe todavia.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
CACHE_CSV = NOTEBOOK_DIR / "data" / "lecturas_limpias.csv"
GAP_CUTOFF_S = 300

# --- Carga desde cache (post-dedup, generado por 01_caracterizacion_fondo.ipynb) ---
# Si no existe, correr ese notebook primero -- el dedup de abril vive ahi, no aca.
df = pd.read_csv(CACHE_CSV)
df["device_id"] = df["device_id"].astype("category")
df["device_code"] = df["device_code"].astype("category")
df["ts"] = pd.to_datetime(df["ts"], format="ISO8601", utc=True)

df["delta_peso"] = df.groupby("device_id", observed=True)["peso"].diff()
df["delta_t"] = df.groupby("device_id", observed=True)["ts"].diff().dt.total_seconds()
df["abs_delta_peso"] = df["delta_peso"].abs()
is_gap = df["delta_t"] > GAP_CUTOFF_S
paso_estable = (df["delta_peso"] == 0) & (~is_gap.fillna(False))

print(f"Lecturas totales (post-dedup, desde cache): {len(df):,}")
print(df["device_code"].value_counts())


## Construccion v1 (ingenua): corta en la primera lectura sin cambio

```
es_movimiento = delta_peso.notna() & (delta_peso != 0) & (~is_gap)
```

El `& (~is_gap)` es necesario: sin el, la primera lectura despues de un
corte largo se marcaria como "movimiento" solo por el salto de tiempo, no
por actividad real.


In [ ]:
es_movimiento_v1 = df["delta_peso"].notna() & (df["delta_peso"] != 0) & (~is_gap.fillna(False))
cid_v1 = (~es_movimiento_v1).groupby(df["device_code"], observed=True).cumsum()

def construir_segmentos(mask, cid, device_codes):
    filas = []
    for _code in device_codes:
        _mask = mask & (df["device_code"] == _code)
        if not _mask.any():
            continue
        _grp = df.loc[_mask].groupby(cid[_mask], observed=True)
        _seg = pd.DataFrame({
            "device_code": _code,
            "n_lecturas": _grp.size(),
            "duracion_s": _grp["delta_t"].sum(),
            "delta_neto_g": _grp["delta_peso"].sum(),
            "max_abs_delta_g": _grp["abs_delta_peso"].max(),
            "idx_inicio": _grp.apply(lambda g: g.index.min()),
            "idx_fin": _grp.apply(lambda g: g.index.max()),
        })
        filas.append(_seg)
    _out = pd.concat(filas, ignore_index=True)
    _out["direccion"] = np.where(_out["delta_neto_g"] > 0, "sube", "baja")
    return _out

DEVICE_CODES = df["device_code"].cat.categories
segmentos_v1_df = construir_segmentos(es_movimiento_v1, cid_v1, DEVICE_CODES)
print(f"Segmentos activos (v1, sin tolerancia): {len(segmentos_v1_df):,}")
print(segmentos_v1_df.groupby("device_code", observed=True).size())


## Hallazgo 6 - cortar por una sola lectura de fondo parte eventos reales en dos

Con la resolucion de este sensor, `delta_peso == 0` es ~98% de la senial
(Paso 1) - una pausa de una sola lectura DENTRO de un evento real de
alimentacion (el animal se detiene un instante) es estadisticamente
esperable, no un caso raro. La v1 corta ahi y parte un evento fisicamente
continuo en dos segmentos, cada uno con un maximo menor al del evento
completo -- riesgo de falso negativo justo en los eventos que mas importan.

**Chequeo:** ¿cuantos pares de corridas de movimiento consecutivas estan
separadas por exactamente 1 lectura, y esa lectura es realmente
`delta_peso==0` genuino (no un gap, no un NaN)?


In [ ]:
for _code in DEVICE_CODES:
    _mask = df["device_code"] == _code
    _sub = df.loc[_mask].copy()
    _sub["es_mov"] = es_movimiento_v1[_mask].values
    _sub["cid"] = cid_v1[_mask].values
    _ids_en_orden = _sub.loc[_sub["es_mov"], "cid"].drop_duplicates().tolist()
    _diffs = np.diff(_ids_en_orden)
    _n_separados_por_1 = int((_diffs == 1).sum())
    _separadores_id = [_ids_en_orden[i] + 1 for i in range(len(_ids_en_orden) - 1) if _diffs[i] == 1]
    _filas_sep = _sub[_sub["cid"].isin(_separadores_id) & ~_sub["es_mov"]]
    _es_pausa_real = (_filas_sep["delta_peso"] == 0).sum()
    print(f"--- {_code} ---")
    print(f"corridas de movimiento totales: {len(_ids_en_orden):,}")
    print(f"pares separados por exactamente 1 lectura: {_n_separados_por_1:,} "
          f"({_n_separados_por_1 / len(_ids_en_orden) * 100:.1f}% de las corridas)")
    print(f"de esas, la lectura separadora es pausa real (delta_peso==0): "
          f"{_es_pausa_real:,} / {len(_filas_sep):,}")
    print()


## Corrección: tolerar 1 lectura aislada de fondo dentro de un segmento activo

Mismo tipo de ajuste que `GAP_CUTOFF_S` (que evita confundir un corte largo
con estabilidad real) pero simétrico, del lado de no cortar de más un evento
activo por una pausa breve: un corte real ahora requiere `is_gap`, un `NaN`,
o **2 o más** lecturas consecutivas de `paso_estable` -- una sola lectura de
`paso_estable` aislada entre movimiento se absorbe dentro del mismo
segmento.


In [ ]:
cambio = (paso_estable != paso_estable.shift(1)) | (df["device_code"] != df["device_code"].shift(1))
racha_id = cambio.cumsum()
racha_len = df.groupby(racha_id)["peso"].transform("size")
es_corte_real = is_gap.fillna(False) | df["delta_peso"].isna() | (paso_estable & (racha_len >= 2))
es_movimiento_v2 = ~es_corte_real
cid_v2 = es_corte_real.groupby(df["device_code"], observed=True).cumsum()

segmentos_v2_df = construir_segmentos(es_movimiento_v2, cid_v2, DEVICE_CODES)

def resumen_candidatos(seg_df, nombre):
    filas = []
    for _code, _grupo in seg_df.groupby("device_code", observed=True):
        _mediana = _grupo["max_abs_delta_g"].median()
        _mad = (_grupo["max_abs_delta_g"] - _mediana).abs().median()
        _umbral = _mediana + (3.5 / 0.6745) * _mad
        _n_cand = (_grupo["max_abs_delta_g"] > _umbral).sum()
        filas.append({
            "version": nombre, "device_code": _code, "n_segmentos": len(_grupo),
            "mediana_g": round(_mediana, 2), "mad_g": round(_mad, 2),
            "umbral_g": round(_umbral, 2), "n_candidatos": int(_n_cand),
        })
    return filas

comparacion_v1_v2 = pd.DataFrame(
    resumen_candidatos(segmentos_v1_df, "v1_sin_tolerancia") +
    resumen_candidatos(segmentos_v2_df, "v2_tolera_1_pausa")
).set_index(["device_code", "version"])
comparacion_v1_v2


## Lectura del Hallazgo 6

**Resultado real (2026-08-29):** KPCL0034 pasa de 2,903 a 2,324 segmentos
(580 pares fusionados, como predecía el chequeo) y los candidatos **suben**
de 462 a 491 (+29). KPCL0035 pasa de 1,243 a 1,084 segmentos y los
candidatos **bajan** de 138 a 132 (−6).

Que KPCL0035 baje no es un problema — son **dos efectos mezclados**, ambos
correctos:
1. **Recuperar eventos partidos**: un evento real dividido en dos mitades,
   cada una bajo el umbral por separado, se fusiona y su máximo combinado sí
   supera el umbral — esto only puede sumar candidatos.
2. **Deduplicar eventos ya contados dos veces**: si dos corridas que YA eran
   candidatas por separado resultan ser el mismo evento físico partido por
   una pausa, ahora se cuentan como **una sola** — esto solo puede restar.

En KPCL0034 domina el efecto 1 (neto +29). En KPCL0035 domina el efecto 2
(neto −6). Ambos son la corrección funcionando bien, no un problema nuevo.

**Decisión:** `segmentos_v2_df` (con tolerancia) queda como la definición
final de este Paso 3 — se descarta `segmentos_v1_df`, que subestimaba
algunos eventos reales por un artefacto de corte, tal como identificó la
revisión de este notebook.


In [ ]:
segmentos_df = segmentos_v2_df
print(f"Segmentos activos finales: {len(segmentos_df):,}")
print(segmentos_df.groupby("device_code", observed=True).size())


## Caracterizacion de los segmentos activos (final, sin filtrar todavia)

Igual que Paso 1 con la senial cruda: primero se describe la distribucion
completa, recien despues se decide un criterio de corte.


In [ ]:
pct = [50, 75, 90, 95, 99]
for _code, _grupo in segmentos_df.groupby("device_code", observed=True):
    print(f"--- {_code} (n={len(_grupo):,}) ---")
    for _col in ("duracion_s", "delta_neto_g", "max_abs_delta_g", "n_lecturas"):
        _dist = _grupo[_col].abs().quantile([p / 100 for p in pct])
        print(f"  {_col} (abs) P50/75/90/95/99: {_dist.round(2).tolist()}")
    print()


## Filtro de candidatos: z-score modificado sobre `max_abs_delta_g`

`max_abs_delta_g` (el salto mas grande dentro del segmento) es la variable
elegida para el corte -- a diferencia de `delta_neto_g`, no se cancela si el
plato sube y baja dentro del mismo segmento activo.

Sobre la poblacion de segmentos activos (no sobre lecturas individuales -
eso ya se descarto en el Hallazgo 4 de Baseline v0/v1), se aplica el
**z-score modificado** de Iglewicz & Hoaglin: `0.6745 * (x - mediana) / MAD`,
con el umbral estandar de la literatura `|z| > 3.5` -- equivale a marcar
candidato cuando `x > mediana + 5.19 * MAD`.


In [ ]:
filas_candidatos = []
for _code, _grupo in segmentos_df.groupby("device_code", observed=True):
    _mediana = _grupo["max_abs_delta_g"].median()
    _mad = (_grupo["max_abs_delta_g"] - _mediana).abs().median()
    _umbral = _mediana + (3.5 / 0.6745) * _mad  # z modificado > 3.5 (Iglewicz & Hoaglin)
    _es_candidato = _grupo["max_abs_delta_g"] > _umbral
    print(f"--- {_code} ---")
    print(f"mediana={_mediana:.2f}g  MAD={_mad:.2f}g  umbral (z_mod>3.5)={_umbral:.2f}g")
    print(f"candidatos: {_es_candidato.sum():,} de {len(_grupo):,} "
          f"({_es_candidato.mean() * 100:.1f}%)")
    print()

    filas_candidatos.append({
        "device_code": _code, "mediana_g": round(_mediana, 2), "mad_g": round(_mad, 2),
        "umbral_g": round(_umbral, 2), "n_candidatos": int(_es_candidato.sum()),
        "n_segmentos": len(_grupo), "pct_candidatos": round(_es_candidato.mean() * 100, 1),
    })
    segmentos_df.loc[_grupo.index, "es_candidato"] = _es_candidato

candidatos_resumen_df = pd.DataFrame(filas_candidatos).set_index("device_code")
candidatos_resumen_df


## Cierre del Paso 3

**Resultado real (2026-08-29, con la correccion del Hallazgo 6 ya aplicada):**

| device_code | segmentos activos | candidatos | % |
|---|---|---|---|
| KPCL0034 | 2,324 | 491 | ~21% |
| KPCL0035 | 1,084 | 132 | ~12% |

**Nota de orientacion, no de validacion:** `Knowledge/09_Sensores/README_Sensores.md`
documenta 421 candidatos detectados para KPCL0034 con el pipeline viejo, sobre
un periodo similar. El numero de aca (491) sigue en el mismo orden de
magnitud, pero **no es una comparacion valida** - son metodos completamente
distintos y esta linea de trabajo no se apoya en el pipeline viejo para nada.

**Lo que este Paso 3 SI logra:** una lista de candidatos a evento (segmentos
activos, con duracion/delta_neto/direccion), derivada 100% de las
estadisticas propias de Investigacion_v2, sin ninguna dependencia del motor
real, y con un corte de segmento ya corregido para no partir eventos por
una pausa aislada (Hallazgo 6).

**Lo que este Paso 3 NO hace:** clasificar los candidatos en
alimentacion/servido/ruido -- eso requeriria features de forma, que es
exactamente lo que esta linea de trabajo decidio no reusar del motor viejo.


## Contexto pre/post — preparar candidatos para revisión manual

El límite mecánico del segmento activo no alcanza para saber si un candidato es
**alimentación** (el plato pesa menos al terminar), **servido** (pesa más), o
**ruido/error** (vuelve a ~lo mismo, aunque haya habido movimiento grande en el
medio — indistinguible de un evento real si solo se mira la magnitud interna
del segmento). Para eso hace falta el nivel **antes** y **después** del segmento,
no solo lo que pasó dentro.

`nivel_antes`/`nivel_despues` = mediana de las `K=5` lecturas `paso_estable`
inmediatamente antes/después del segmento (búsqueda por índice ordenado, no
por tiempo — más rápido y exacto). `K=5` a ~30s de cadencia es ~2.5 min de
margen, chico frente a la duración típica de una corrida estable (mediana
500-660s, Paso 1) — no debería chocar con otro evento en la mayoría de los
casos. Si no hay 5 lecturas estables disponibles de un lado (borde de los
datos o cerca de un gap), se marca `sin_contexto=True` en vez de forzar un
cálculo con datos que no existen.

**Sin motor viejo y sin anotaciones, no hay ninguna etiqueta para validar una
regla de clasificación automática contra algo real** — por eso esto no
construye una regla, prepara los datos para que la revisión sea **manual**:
tabla exportada + función para graficar cualquier candidato con su curva real.


In [ ]:
K_MARGEN = 5

niveles_antes, niveles_despues = [], []
for _code, _sub in segmentos_df.groupby("device_code", observed=True):
    _idx_estable = np.array(sorted(df.index[(df["device_code"] == _code) & paso_estable].tolist()))
    for _, _row in _sub.iterrows():
        _pi = np.searchsorted(_idx_estable, _row["idx_inicio"])
        _antes = _idx_estable[max(0, _pi - K_MARGEN):_pi]
        _pf = np.searchsorted(_idx_estable, _row["idx_fin"], side="right")
        _despues = _idx_estable[_pf:_pf + K_MARGEN]
        niveles_antes.append(df.loc[_antes, "peso"].median() if len(_antes) == K_MARGEN else np.nan)
        niveles_despues.append(df.loc[_despues, "peso"].median() if len(_despues) == K_MARGEN else np.nan)

segmentos_df["nivel_antes"] = niveles_antes
segmentos_df["nivel_despues"] = niveles_despues
segmentos_df["delta_neto_real"] = segmentos_df["nivel_despues"] - segmentos_df["nivel_antes"]
segmentos_df["sin_contexto"] = segmentos_df["nivel_antes"].isna() | segmentos_df["nivel_despues"].isna()
segmentos_df["ts_inicio"] = df.loc[segmentos_df["idx_inicio"], "ts"].values
segmentos_df["ts_fin"] = df.loc[segmentos_df["idx_fin"], "ts"].values

print(f"Segmentos sin contexto suficiente (borde/gap): "
      f"{segmentos_df['sin_contexto'].sum():,} de {len(segmentos_df):,} "
      f"({segmentos_df['sin_contexto'].mean() * 100:.2f}%)")


## Exportar candidatos para revisión manual

`data/candidatos_para_revision.csv` — uno por fila, con contexto antes/después
y timestamps legibles. Orden cronológico dentro de cada `device_code`, para
poder ir mirándolos en el orden en que pasaron.


In [ ]:
COLUMNAS_REVISION = [
    "device_code", "idx_inicio", "idx_fin", "ts_inicio", "ts_fin", "duracion_s", "n_lecturas",
    "nivel_antes", "nivel_despues", "delta_neto_real", "max_abs_delta_g",
    "es_candidato", "sin_contexto",
]
candidatos_revision_df = (
    segmentos_df.loc[segmentos_df["es_candidato"], COLUMNAS_REVISION]
    .sort_values(["device_code", "ts_inicio"])
    .reset_index(drop=True)
)
CANDIDATOS_CSV = NOTEBOOK_DIR / "data" / "candidatos_para_revision.csv"
candidatos_revision_df.to_csv(CANDIDATOS_CSV, index=False)
print(f"Exportado: {CANDIDATOS_CSV} ({len(candidatos_revision_df):,} candidatos)")
candidatos_revision_df.head(10)


## Graficar un candidato con su curva real (revisión manual)

`graficar_candidato(fila)` dibuja la curva completa (segmento + margen de
contexto) con el tramo del segmento sombreado. `graficar_pagina(pagina, n)`
muestra varios a la vez para revisar más rápido. Cambiar `pagina` para ver
los siguientes `n` candidatos.


In [ ]:
import matplotlib.pyplot as plt

MARGEN_GRAFICO_LECTURAS = 10  # un poco mas ancho que K_MARGEN, solo para visualizar


def graficar_candidato(fila, ax=None):
    _m = df["device_code"] == fila["device_code"]
    _ini = max(df.loc[_m].index.min(), fila["idx_inicio"] - MARGEN_GRAFICO_LECTURAS)
    _fin = min(df.loc[_m].index.max(), fila["idx_fin"] + MARGEN_GRAFICO_LECTURAS)
    _ventana = df.loc[_m].loc[_ini:_fin]
    _ax = ax or plt.subplots(figsize=(6, 2.8))[1]
    _ax.plot(_ventana["ts"], _ventana["peso"], marker="o", markersize=3)
    _ax.axvspan(fila["ts_inicio"], fila["ts_fin"], color="orange", alpha=0.2)
    _ax.set_title(
        f"{fila['device_code']} {fila['ts_inicio']:%m-%d %H:%M} | "
        f"dur={fila['duracion_s']:.0f}s | neto_real={fila['delta_neto_real']:.1f}g",
        fontsize=9,
    )
    _ax.tick_params(axis="x", labelrotation=20, labelsize=7)
    return _ax


def graficar_pagina(pagina=0, n=9, device_code=None):
    _df = candidatos_revision_df
    if device_code:
        _df = _df[_df["device_code"] == device_code]
    _lote = _df.iloc[pagina * n:(pagina + 1) * n]
    _cols = 3
    _filas = -(-len(_lote) // _cols)
    _fig, _axes = plt.subplots(_filas, _cols, figsize=(5 * _cols, 2.8 * _filas))
    for _ax, (_, _fila) in zip(np.atleast_1d(_axes).flatten(), _lote.iterrows()):
        graficar_candidato(_fila, ax=_ax)
    for _ax in np.atleast_1d(_axes).flatten()[len(_lote):]:
        _ax.axis("off")
    _fig.tight_layout()
    return _fig


# Ejemplo: primeros 9 candidatos de KPCL0034, en orden cronologico
graficar_pagina(pagina=0, n=9, device_code="KPCL0034")
plt.show()


## Etiquetado manual — revisión persistida

`etiquetar(device_code, ts_inicio, categoria)` guarda la decisión manual en
`data/candidatos_etiquetas_manuales.csv` (identificado por `device_code` +
`ts_inicio`, no por posición — así no se rompe si cambia el orden de una
página a otra). Categorías: `"alimentacion"`, `"servido"`, `"ruido"`.

**Duración de referencia (dato del dominio, no derivado de los datos):**
servido dura 1-8s (más rápido que nuestra cadencia de 30s — siempre se ve
como 1 sola lectura), alimentación dura ~3-8 min (180-480s, ~6-16 lecturas).


In [ ]:
LABELS_CSV = NOTEBOOK_DIR / "data" / "candidatos_etiquetas_manuales.csv"
if LABELS_CSV.exists():
    etiquetas_df = pd.read_csv(LABELS_CSV, parse_dates=["ts_inicio"])
else:
    etiquetas_df = pd.DataFrame(columns=["device_code", "ts_inicio", "categoria"])


def etiquetar(device_code, ts_inicio, categoria):
    global etiquetas_df
    ts_inicio = pd.Timestamp(ts_inicio)
    etiquetas_df = etiquetas_df[
        ~((etiquetas_df["device_code"] == device_code) & (etiquetas_df["ts_inicio"] == ts_inicio))
    ]
    _nueva = pd.DataFrame([{"device_code": device_code, "ts_inicio": ts_inicio, "categoria": categoria}])
    etiquetas_df = _nueva if etiquetas_df.empty else pd.concat([etiquetas_df, _nueva], ignore_index=True)
    etiquetas_df.to_csv(LABELS_CSV, index=False)


# Revision manual 2026-08-29 -- confirmado: 03:15/03:17/03:19 oscilan (sube-baja-
# sube en 4 min), firma de manipulacion/disturbio, no de un servido real (que es
# un salto UNICO y aislado que se queda estable). 23:25 y 02:32 tambien ruido.
for _ts in [
    "2026-04-08 03:15:13.448390", "2026-04-08 03:17:43.867792",
    "2026-04-08 03:19:13.667483", "2026-04-08 23:25:54.371354",
    "2026-04-09 02:32:28.644000",
]:
    etiquetar("KPCL0034", _ts, "ruido")

print(f"Etiquetas guardadas: {len(etiquetas_df):,}")
etiquetas_df


## Siguiente página de revisión


In [ ]:
graficar_pagina(pagina=1, n=9, device_code="KPCL0034")
plt.show()


## Resumen final (tabla y gráfico)


In [ ]:
import matplotlib.pyplot as plt

resumen_final_df = candidatos_resumen_df
print(resumen_final_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
_x = np.arange(len(resumen_final_df))

# Panel 1: segmentos activos totales vs candidatos
axes[0].bar(_x - 0.2, resumen_final_df["n_segmentos"], width=0.4, label="Segmentos activos", color="#7f8c8d")
axes[0].bar(_x + 0.2, resumen_final_df["n_candidatos"], width=0.4, label="Candidatos", color="#c0392b")
axes[0].set_xticks(_x)
axes[0].set_xticklabels(resumen_final_df.index)
axes[0].set_ylabel("Cantidad")
axes[0].set_title("Segmentos activos vs candidatos")
axes[0].legend()

# Panel 2: umbral vs mediana (donde corta el z-score modificado)
axes[1].bar(_x - 0.2, resumen_final_df["mediana_g"], width=0.4, label="Mediana", color="#2980b9")
axes[1].bar(_x + 0.2, resumen_final_df["umbral_g"], width=0.4, label="Umbral (z_mod>3.5)", color="#e67e22")
axes[1].set_xticks(_x)
axes[1].set_xticklabels(resumen_final_df.index)
axes[1].set_ylabel("max_abs_delta_g")
axes[1].set_title("Mediana vs umbral de candidato")
axes[1].legend()

# Panel 3: distribucion de max_abs_delta_g por segmento, con el umbral marcado
for _i, (_code, _grupo) in enumerate(segmentos_df.groupby("device_code", observed=True)):
    axes[2].hist(_grupo["max_abs_delta_g"], bins=30, range=(0, 60), alpha=0.5, label=_code)
    axes[2].axvline(resumen_final_df.loc[_code, "umbral_g"], linestyle="--", color=f"C{_i}")
axes[2].set_xlabel("max_abs_delta_g por segmento")
axes[2].set_ylabel("Cantidad de segmentos")
axes[2].set_title("Distribucion + umbral de candidato")
axes[2].set_xlim(0, 60)
axes[2].set_yscale("log")
axes[2].legend()

fig.suptitle("Paso 3 - Resumen: deteccion de candidatos desde cero")
fig.tight_layout()
plt.show()
